# Proyecto final de Optimización
## Expansión de la capacidad de computación (2027–2029)

**Resultado del caso base.** Se activan Madrid, Eemshaven, Columbus y Dallas, con 29 nodos en total. El coste mínimo es de 16,206904 M€.

Se decide qué centros activar, cuántos nodos instalar de cada tecnología y cómo asignar los 7.000 HEH diarios de demanda. El objetivo es minimizar la suma de los costes de activación, compra de nodos y electricidad durante el horizonte.

La formulación procede de **Formulacion_Proyecto_Optimizacion_final.pdf** (p. 1). Los valores numéricos de tecnologías, demanda, centros y latencia se toman de **Proyecto_Optimizacion_OpenAI.docx** (tablas 1 a 7 del enunciado original). Se usan los códigos de mercados del PDF adjunto (SP para España y GE para Alemania).

El cuaderno usa Python, Pyomo, pandas y Gurobi mediante *gurobi_direct*. Los datos están incluidos en las celdas.

## 1. Conjuntos y datos

$L$ es el conjunto de centros, $T$ el de tecnologías, $M$ el de mercados y $A$ el de áreas. El conjunto $E$ contiene los pares $(m,l)$ con latencia de hasta 60 ms, incluido el límite.

Se usan los 1.095 días especificados en la formulación. Los coeficientes $EH_t$ son los valores publicados en el enunciado y ya incluyen el PUE. La capacidad diaria se calcula al 100 % a partir de la capacidad horaria y el límite del 80 % se aplica en la restricción de capacidad.

In [1]:
import pyomo.environ as pe
import pyomo.opt as po
import pandas as pd
from IPython.display import display

model = pe.ConcreteModel()

model.L = pe.Set(initialize=["MAD", "PAR", "FRA", "LON", "EEM", "ASH", "COL", "DAL"])
model.T = pe.Set(initialize=["H100", "H200", "B200"])
model.M = pe.Set(initialize=["SP", "FR", "GE", "UK", "BE", "US.E", "US.C", "US.W"])
model.A = pe.Set(initialize=["EU", "US"])

# Centro: área, coste fijo (M€), máx. nodos, potencia IT (kW), electricidad (€/kWh)
centros = {
    "MAD": ("EU", 1.20, 16, 180, 0.128),
    "PAR": ("EU", 1.80, 18, 220, 0.153),
    "FRA": ("EU", 2.20, 24, 300, 0.2264),
    "LON": ("EU", 2.10, 18, 220, 0.279),
    "EEM": ("EU", 1.60, 24, 300, 0.199),
    "ASH": ("US", 2.00, 30, 380, 0.0856),
    "COL": ("US", 1.40, 24, 300, 0.0931),
    "DAL": ("US", 1.50, 32, 400, 0.0601),
}

# Tecnología: disponibilidad, coste (M€/nodo), potencia (kW/nodo), HEH/h, kWh/HEH
tecnologias = {
    "H100": (24, 0.260, 10.2, 8.0, 1.530),
    "H200": (18, 0.300, 10.2, 9.2, 1.330),
    "B200": (12, 0.390, 14.3, 18.0, 0.953),
}

demanda = {
    "SP": 560, "FR": 700, "GE": 1_050, "UK": 840, "BE": 560,
    "US.E": 1_260, "US.C": 910, "US.W": 1_120,
}

# RTT (ms). Columnas: SP, FR, GE, UK, BE, US.E, US.C, US.W.
latencia_por_centro = {
    "MAD": (5, 21, 27, 24, 27, 92, 106, 145),
    "PAR": (21, 5, 12, 10, 11, 93, 105, 141),
    "FRA": (27, 12, 5, 15, 10, 98, 109, 145),
    "LON": (24, 10, 15, 5, 10, 89, 100, 136),
    "EEM": (30, 14, 11, 13, 8, 94, 105, 139),
    "ASH": (97, 98, 103, 94, 98, 10, 19, 60),
    "COL": (102, 102, 107, 98, 102, 16, 12, 53),
    "DAL": (124, 124, 129, 120, 124, 38, 24, 35),
}
latencia = {
    (m, l): rtt
    for l, fila in latencia_por_centro.items()
    for m, rtt in zip(model.M, fila, strict=True)
}

model.TDD = pe.Param(initialize=7_000)
model.alpha = pe.Param(initialize=0.70)
model.MOU = pe.Param(initialize=0.80)
model.MIB = pe.Param(initialize=18)
model.ND = pe.Param(initialize=1_095)
latencia_maxima = 60
nodos_minimos = 4
centros_minimos = 2

assert sum(demanda.values()) == pe.value(model.TDD)

model.DD = pe.Param(model.M, initialize=demanda)
model.LA = pe.Param(model.M, model.L, initialize=latencia)
model.FK = pe.Param(model.L, initialize={l: centros[l][1] for l in model.L})
model.MN = pe.Param(model.L, initialize={l: centros[l][2] for l in model.L})
model.AP = pe.Param(model.L, initialize={l: centros[l][3] for l in model.L})
model.EP = pe.Param(model.L, initialize={l: centros[l][4] for l in model.L})
model.B = pe.Param(
    model.L, model.A,
    initialize={(l, a): int(centros[l][0] == a) for l in model.L for a in model.A},
)
model.AV = pe.Param(model.T, initialize={t: tecnologias[t][0] for t in model.T})
model.UK = pe.Param(model.T, initialize={t: tecnologias[t][1] for t in model.T})
model.MP = pe.Param(model.T, initialize={t: tecnologias[t][2] for t in model.T})
model.OQ = pe.Param(model.T, initialize={t: 24 * tecnologias[t][3] for t in model.T})
model.EH = pe.Param(model.T, initialize={t: tecnologias[t][4] for t in model.T})

model.E = pe.Set(
    dimen=2, within=model.M * model.L,
    initialize=[(m, l) for m in model.M for l in model.L if model.LA[m, l] <= latencia_maxima],
)

## 2. Variables y función objetivo

- $X_l \in \{0,1\}$ indica si se activa el centro $l$.
- $Q_{lt} \in \mathbb{Z}_{\geq 0}$ es el número de nodos de tecnología $t$ instalados en $l$.
- $W_{mlt} \geq 0$ son los HEH diarios del mercado $m$ procesados en $l$ con $t$. Se define para los pares de $E$.

El CAPEX suma los costes fijos y de hardware en M€. En electricidad, $EP_l EH_t W_{mlt}$ da €/día. Se multiplica por $ND$ y se divide por $10^6$ para obtener M€ del horizonte.

In [2]:
model.X = pe.Var(model.L, within=pe.Binary)
model.Q = pe.Var(model.L, model.T, within=pe.NonNegativeIntegers)
model.W = pe.Var(model.E, model.T, within=pe.NonNegativeReals)


def coste_fijo_rule(model):
    return sum(model.FK[l] * model.X[l] for l in model.L)


def coste_hardware_rule(model):
    return sum(model.UK[t] * model.Q[l, t] for l in model.L for t in model.T)


def coste_electricidad_rule(model):
    return model.ND / 1_000_000 * sum(
        model.EP[l] * model.EH[t] * model.W[m, l, t]
        for m, l in model.E for t in model.T
    )


model.coste_fijo = pe.Expression(rule=coste_fijo_rule)
model.coste_hardware = pe.Expression(rule=coste_hardware_rule)
model.coste_electricidad = pe.Expression(rule=coste_electricidad_rule)
model.capex = pe.Expression(expr=model.coste_fijo + model.coste_hardware)


def objetivo_rule(model):
    return model.capex + model.coste_electricidad


model.objetivo = pe.Objective(rule=objetivo_rule, sense=pe.minimize)

## 3. Restricciones

Se implementan las restricciones 2 a 10 de la formulación en el mismo orden. Los dominios de la restricción 1 se declaran en las variables y el umbral de latencia se aplica al construir $E$.

La cobertura se exige para cada mercado. El límite del 70 % agrega todas las tecnologías de un mismo centro. El presupuesto de 18 M€ se aplica al CAPEX y la electricidad se suma en el objetivo.

In [3]:
def demanda_rule(model, m):
    carga = sum(
        model.W[m, l, t] for l in model.L for t in model.T if (m, l) in model.E
    )
    return carga == model.DD[m]


def concentracion_rule(model, m, l):
    return sum(model.W[m, l, t] for t in model.T) <= model.alpha * model.DD[m]


def capacidad_rule(model, l, t):
    carga = sum(model.W[m, l, t] for m in model.M if (m, l) in model.E)
    return carga <= model.MOU * model.OQ[t] * model.Q[l, t]


def nodos_minimos_rule(model, l):
    return sum(model.Q[l, t] for t in model.T) >= nodos_minimos * model.X[l]


def nodos_maximos_rule(model, l):
    return sum(model.Q[l, t] for t in model.T) <= model.MN[l] * model.X[l]


def disponibilidad_rule(model, t):
    return sum(model.Q[l, t] for l in model.L) <= model.AV[t]


def potencia_rule(model, l):
    return sum(model.MP[t] * model.Q[l, t] for t in model.T) <= model.AP[l] * model.X[l]


def centros_por_area_rule(model, a):
    return sum(model.B[l, a] * model.X[l] for l in model.L) >= centros_minimos


def presupuesto_rule(model):
    return model.capex <= model.MIB


model.demanda = pe.Constraint(model.M, rule=demanda_rule)
model.concentracion = pe.Constraint(model.E, rule=concentracion_rule)
model.capacidad = pe.Constraint(model.L, model.T, rule=capacidad_rule)
model.nodos_minimos = pe.Constraint(model.L, rule=nodos_minimos_rule)
model.nodos_maximos = pe.Constraint(model.L, rule=nodos_maximos_rule)
model.disponibilidad = pe.Constraint(model.T, rule=disponibilidad_rule)
model.potencia = pe.Constraint(model.L, rule=potencia_rule)
model.centros_por_area = pe.Constraint(model.A, rule=centros_por_area_rule)
model.presupuesto = pe.Constraint(rule=presupuesto_rule)

## 4. Resolución y comprobación

In [4]:
solver = po.SolverFactory("gurobi_direct")
solver.options["MIPGap"] = 0
results = solver.solve(model, tee=False)
po.assert_optimal_termination(results)

tolerancia = 1e-6
for restriccion in model.component_data_objects(pe.Constraint, active=True):
    valor = pe.value(restriccion.body)
    if restriccion.has_lb():
        assert valor >= pe.value(restriccion.lower) - tolerancia, restriccion.name
    if restriccion.has_ub():
        assert valor <= pe.value(restriccion.upper) + tolerancia, restriccion.name

for variable in model.component_data_objects(pe.Var, active=True):
    valor = pe.value(variable)
    assert valor >= -tolerancia, variable.name
    if variable.is_integer():
        assert abs(valor - round(valor)) <= tolerancia, variable.name
    if variable.is_binary():
        assert valor <= 1 + tolerancia, variable.name

assert all(model.LA[m, l] <= latencia_maxima for m, l, t in model.W)

objetivo = pe.value(model.objetivo)
cota = results.problem.lower_bound
assert abs(objetivo - cota) <= tolerancia
print(f"Terminación: {results.solver.termination_condition}")
print(f"Cota inferior de Gurobi: {cota:.6f} M€")
print(f"Coste total mínimo: {objetivo:.6f} M€")
print(f"Pares admisibles: {len(model.E)} de {len(model.M) * len(model.L)}")
print("Comprobados los dominios, la latencia y todas las restricciones del modelo.")

Terminación: optimal
Cota inferior de Gurobi: 16.206904 M€
Coste total mínimo: 16.206904 M€
Pares admisibles: 34 de 64
Comprobados los dominios, la latencia y todas las restricciones del modelo.


## 5. Resultados

La tabla incluye todos los centros. La potencia IT se calcula a partir de los nodos instalados y la carga corresponde al trabajo asignado en HEH/día.

In [5]:
filas_centros = []
for l in model.L:
    nodos = {t: round(pe.value(model.Q[l, t])) for t in model.T}
    filas_centros.append({
        "Centro": l,
        "Área": centros[l][0],
        "Activo": "Sí" if round(pe.value(model.X[l])) else "No",
        **nodos,
        "Nodos": sum(nodos.values()),
        "Potencia IT (kW)": sum(model.MP[t] * nodos[t] for t in model.T),
        "Carga (HEH/día)": sum(
            pe.value(model.W[m, l, t]) for m in model.M if (m, l) in model.E
            for t in model.T
        ),
    })

resumen_centros = pd.DataFrame(filas_centros).set_index("Centro")
display(resumen_centros.round(2))

,Área,Activo,H100,H200,B200,Nodos,Potencia IT (kW),Carga (HEH/día)
Centro,,,,,,,,
MAD,EU,Sí,4,3,4,11,128.6,2526.72
PAR,EU,No,0,0,0,0,0.0,0.00
FRA,EU,No,0,0,0,0,0.0,0.00
LON,EU,No,0,0,0,0,0.0,0.00
EEM,EU,Sí,1,0,3,4,53.1,1183.28
ASH,US,No,0,0,0,0,0.0,0.00
COL,US,Sí,1,1,2,4,49.0,1016.72
DAL,US,Sí,0,7,3,10,114.3,2273.28


### Reparto de la demanda

Cada fila muestra una asignación mercado–centro con carga positiva. Las columnas de las tres tecnologías y el total están en HEH/día. La columna de cuota indica el porcentaje de la demanda del mercado servido por ese centro.

In [6]:
filas_reparto = []
for m, l in model.E:
    carga = {t: pe.value(model.W[m, l, t]) for t in model.T}
    total = sum(carga.values())
    if total > tolerancia:
        filas_reparto.append({
            "Mercado": m,
            "Centro": l,
            **carga,
            "Total": total,
            "Cuota (%)": 100 * total / demanda[m],
            "RTT (ms)": model.LA[m, l],
        })

reparto = pd.DataFrame(filas_reparto).set_index(["Mercado", "Centro"])
display(reparto.round(3))

H100    H200    B200   Total  Cuota (%)  RTT (ms)
Mercado Centro                                                     
SP      MAD      62.20  264.96   29.70  356.86     63.725         5
        EEM       0.00    0.00  203.14  203.14     36.275        30
FR      MAD     490.00    0.00    0.00  490.00     70.000        21
        EEM       0.00    0.00  210.00  210.00     30.000        14
GE      MAD       0.00    0.00  735.00  735.00     70.000        27
        EEM       0.00    0.00  315.00  315.00     30.000        11
UK      MAD       0.00    0.00  588.00  588.00     70.000        24
        EEM     146.48    0.00  105.52  252.00     30.000        13
BE      MAD      62.20  264.96   29.70  356.86     63.725        27
        EEM       0.00    0.00  203.14  203.14     36.275         8
US.E    COL     148.88    0.00  258.84  407.72     32.359        16
        DAL       0.00  452.48  399.80  852.28     67.641        38
US.C    COL       0.00  176.64   96.36  273.00     30.000        12
        DAL       0.00    0.00  637.00  637.00     70.000        24
US.W    COL       0.00    0.00  336.00  336.00     30.000        53
        DAL       0.00  784.00    0.00  784.00     70.000        35

### Desglose de costes

In [7]:
costes = pd.Series({
    "Activación": pe.value(model.coste_fijo),
    "Hardware": pe.value(model.coste_hardware),
    "CAPEX": pe.value(model.capex),
    "Electricidad en 1.095 días": pe.value(model.coste_electricidad),
    "Coste total": pe.value(model.objetivo),
    "Presupuesto de CAPEX disponible": pe.value(model.MIB - model.capex),
}, name="M€")
display(costes.round(6).to_frame())

,M€
Activación,5.700000
Hardware,9.540000
CAPEX,15.240000
Electricidad en 1.095 días,0.966904
Coste total,16.206904
Presupuesto de CAPEX disponible,2.760000


### Interpretación

El despliegue usa 6 nodos H100, 11 H200 y los 12 B200 disponibles. Los cuatro centros cubren los 7.000 HEH/día y cada mercado se reparte entre dos de ellos. El CAPEX asciende a 15,24 M€, con un margen de 2,76 M€ respecto al presupuesto. La electricidad añade 0,966904 M€ durante el horizonte.

La tabla de reparto muestra una asignación óptima. El coste de procesar un HEH en un mismo centro y tecnología es igual para todos los mercados admisibles, por lo que pueden existir otros repartos con el mismo coste.